In [3]:
import os
import re
import glob
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# My settings
WORKDIR = r"C:\kazhydromet"
SEP = ";"
Y_FILE = "Урожайность.csv"
COMMON_PLOTS_DIR = os.path.join(WORKDIR, "regression_common_plots")
COMMON_DATASETS_DIR = os.path.join(WORKDIR, "regression_common_datasets")
COMMON_RESULTS_DIR = os.path.join(WORKDIR, "regression_common_results")
TARGETS = ["yield_wheat", "yield_cereals"]
PRECIP_COL = "Daily_precipitation_total_mm" # precipitation column for sum aggregation

# anti-multicollinearity at the monthly feature level
MULTICOL_SPEARMAN_THRESH = 0.90 
MUST_KEEP = {"Atmospheric_pressure_at_sea_level_hPa"}

# feature selection in regression
SPEARMAN_THRESH = 0.45
MIN_FEATURES = 3
TOPK_FALLBACK = 5
MAX_FEATURES = 5

#validation paraneters
MIN_TRAIN = 7       # rolling
N_SPLITS = 3        # tscv
TEST_SIZE = 1       

# for plots
FIG_DPI = 300
TOPN_FEATURES = 20  # for heatmap 

# district transliter
REGION_NAME_EN = {
    "Жаксы": "Zhaksy",
    "Аршалы": "Arshaly",
    "Коргалжын": "Korgalzhyn",
    "Ерейментау": "Ereimentau",
}


def to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_filename(name: str, maxlen: int = 140) -> str:
    name = str(name)
    name = re.sub(r'[<>:"/\\|?*\n\r\t]+', "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    if len(name) > maxlen:
        name = name[:maxlen]
    return name

def region_to_en(region_key: str) -> str:
    if region_key in REGION_NAME_EN:
        return REGION_NAME_EN[region_key]
    for k, v in REGION_NAME_EN.items():
        if k.lower() in str(region_key).lower():
            return v
    return str(region_key)

def spearman_pairs(dfX: pd.DataFrame, thresh: float = 0.90):
    corr = dfX.corr(method="spearman")
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = (upper.stack()
                  .rename("rho")
                  .reset_index()
                  .rename(columns={"level_0": "feat_1", "level_1": "feat_2"}))
    strong = pairs[pairs["rho"].abs() >= thresh].copy()
    strong["abs_rho"] = strong["rho"].abs()
    strong = strong.sort_values("abs_rho", ascending=False)
    return corr, strong

def score_feature(name: str, must_keep: set) -> int:
    if name in must_keep:
        return 10_000
    s = 0
    if "average" in name:
        s += 30
    if "maximum" in name:
        s -= 10
    if "minimum" in name:
        s -= 10
    if "at_sea_level" in name:
        s += 20
    if "at_the_station_level" in name:
        s -= 5
    return s

def drop_multicollinear_spearman(X: pd.DataFrame, thresh: float, must_keep: set):
    _, strong = spearman_pairs(X, thresh=thresh)
    to_drop = set()

    for _, r in strong.iterrows():
        f1, f2 = r["feat_1"], r["feat_2"]
        if f1 in to_drop or f2 in to_drop:
            continue
        if score_feature(f1, must_keep) >= score_feature(f2, must_keep):
            to_drop.add(f2)
        else:
            to_drop.add(f1)

    to_drop = [c for c in to_drop if c in X.columns and c not in must_keep]
    return X.drop(columns=to_drop), strong, to_drop

def agg_mean_sum(df_monthly: pd.DataFrame, months: list, suffix: str, precip_col: str) -> pd.DataFrame:
    d = df_monthly[df_monthly["Month"].isin(months)].copy()
    
    present = sorted(d["Month"].unique().tolist())
    missing = sorted(set(months) - set(present))
    if missing:
        print(f"[WARN] {suffix}: отсутствуют месяцы {missing}")

    feat_cols_local = [c for c in d.columns if c not in ["Year", "Month"]]
    non_precip = [c for c in feat_cols_local if c != precip_col]

    out_mean = d.groupby("Year")[non_precip].mean(numeric_only=True)
    out_sum = d.groupby("Year")[[precip_col]].sum(numeric_only=True)

    out = pd.concat([out_mean, out_sum], axis=1)
    out = out.add_suffix(suffix).reset_index()
    return out

def build_models(feature_cols):
    num_linear = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ])
    num_tree = Pipeline([
        ("imp", SimpleImputer(strategy="median"))
    ])

    prep_linear = ColumnTransformer([("num", num_linear, feature_cols)], remainder="drop")
    prep_tree = ColumnTransformer([("num", num_tree, feature_cols)], remainder="drop")

    models = {
        "Linear": Pipeline([("prep", prep_linear), ("model", LinearRegression())]),
        "Ridge": Pipeline([("prep", prep_linear), ("model", Ridge(alpha=1.0, random_state=42))]),
        "Lasso": Pipeline([("prep", prep_linear), ("model", Lasso(alpha=0.01, random_state=42, max_iter=50_000))]),
        "RF": Pipeline([("prep", prep_tree), ("model", RandomForestRegressor(
            n_estimators=600, random_state=42, n_jobs=-1, min_samples_leaf=1))]),
        "GB": Pipeline([("prep", prep_tree), ("model", GradientBoostingRegressor(random_state=42))]),
    }
    return models

def spearman_select(X: pd.DataFrame, y: pd.Series):
    corr = pd.concat([X, y], axis=1).corr(method="spearman")
    rho = corr.loc[X.columns, y.name].abs().dropna()

    selected = rho[rho >= SPEARMAN_THRESH].sort_values(ascending=False).index.tolist()
    if len(selected) < MIN_FEATURES:
        selected = rho.sort_values(ascending=False).head(min(TOPK_FALLBACK, len(rho))).index.tolist()

    selected = selected[:MAX_FEATURES]
    return selected, rho

def get_time_splits(df_len: int, cv_mode: str, min_train: int, n_splits: int, test_size: int):
    if df_len < 3:
        return []

    if cv_mode == "rolling":
        if df_len <= min_train:
            return []
        splits = []
        for t in range(min_train, df_len):
            tr = np.arange(0, t)
            te = np.array([t])
            splits.append((tr, te))
        return splits

    # cv_mode == "tscv"
    n_splits_eff = min(n_splits, df_len - 2)
    if n_splits_eff < 1:
        return []
    tscv = TimeSeriesSplit(n_splits=n_splits_eff, test_size=test_size)
    return list(tscv.split(np.arange(df_len)))

def save_heatmap(corr: pd.DataFrame, title: str, path: str):
    cols = corr.columns.tolist()
    plt.figure(figsize=(10, 8))
    plt.imshow(corr.values, aspect="auto", vmin=-1, vmax=1, cmap='RdBu_r')
    plt.colorbar()
    plt.title(title)
    plt.xticks(range(len(cols)), cols, rotation=90, fontsize=7)
    plt.yticks(range(len(cols)), cols, fontsize=7)
    plt.tight_layout()
    plt.savefig(path, dpi=FIG_DPI)
    plt.close()

def save_bar_absrho(rho: pd.Series, title: str, path: str, topn: int = 25):
    s = rho.abs().sort_values(ascending=False).head(topn)
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(s)), s.values)
    plt.xticks(range(len(s)), s.index, rotation=90, fontsize=7)
    plt.ylabel("|Spearman ρ|")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=FIG_DPI)
    plt.close()

def save_scatter_top(df_num: pd.DataFrame, rho: pd.Series, target: str, prefix_path: str, topk: int = 6):
    top_feats = rho.abs().sort_values(ascending=False).head(topk).index.tolist()
    for feat in top_feats:
        if feat not in df_num.columns or target not in df_num.columns:
            continue
        plt.figure(figsize=(5, 4))
        plt.scatter(df_num[feat], df_num[target], s=14)
        plt.xlabel(feat)
        plt.ylabel(target)
        plt.title(f"{feat} vs {target} (ρ={rho.loc[feat]:.3f})")
        plt.tight_layout()

        feat_safe = safe_filename(feat)
        out_path = f"{prefix_path}_scatter_{feat_safe}.png"
        plt.savefig(out_path, dpi=FIG_DPI)
        plt.close()


def build_tables_for_region(region_key: str, workdir: str, common_datasets_dir: str) -> dict: #build YearTable, WideTable, 2PhaseTable, 3PhaseTable
    os.chdir(workdir)
    final_file = f"{region_key}_Final.csv"
    if not os.path.exists(final_file):
        raise FileNotFoundError(f"File not found: {final_file}")

    df = pd.read_csv(final_file, sep=SEP)

    if "Month_num" in df.columns:
        df = df.rename(columns={"Month_num": "Month"})
    if "Decada_num" in df.columns:
        df = df.drop(columns=["Decada_num"])

    required_cols = {"Year", "Month"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"[{region_key}] no required columns {missing}")

    if PRECIP_COL not in df.columns:
        raise ValueError(f"[{region_key}] no precipitation column '{PRECIP_COL}'")

    feature_cols = [c for c in df.columns if c not in ["Year", "Month"]]
    agg = {c: "mean" for c in feature_cols}
    agg[PRECIP_COL] = "sum"

    monthly = df.groupby(["Year", "Month"], as_index=False).agg(agg)

    feature_cols_monthly = [c for c in monthly.columns if c not in ["Year", "Month"]]
    X = monthly[feature_cols_monthly].copy()
    X = X.apply(pd.to_numeric, errors="coerce")

    all_nan_cols = X.columns[X.isna().all()].tolist()
    if all_nan_cols:
        X = X.drop(columns=all_nan_cols)

    X_clean, strong_pairs, dropped = drop_multicollinear_spearman(
        X, thresh=MULTICOL_SPEARMAN_THRESH, must_keep=MUST_KEEP
    )
    if dropped:
        print(f"[{region_key}]  {len(dropped)} multicollinear features deleted ")

    df_filtered = pd.concat([monthly[["Year", "Month"]], X_clean], axis=1)

    df_filtered["Month"] = pd.to_numeric(df_filtered["Month"], errors="coerce").astype("Int64")
    df_filtered["Year"] = pd.to_numeric(df_filtered["Year"], errors="coerce").astype("Int64")
    
    feat_cols = [c for c in df_filtered.columns if c not in ["Year", "Month"]]
    for c in feat_cols:
        df_filtered[c] = to_float(df_filtered[c])

    yld = pd.read_csv(Y_FILE, sep=SEP)
    if "Region" not in yld.columns or "Year" not in yld.columns:
        raise ValueError("Yield file must have columns Region and Year")

    yld["Region"] = yld["Region"].astype(str).str.strip()
    cols_needed = ["Year"] + TARGETS
    missing_targets = [c for c in cols_needed if c not in yld.columns]
    if missing_targets:
        raise ValueError(f"There are no columns in the yield file: {missing_targets}")

    yld_reg = yld[yld["Region"].str.contains(region_key, case=False, na=False)][cols_needed].copy()
    if yld_reg.empty:
        raise ValueError(f"[{region_key}] district not found in {Y_FILE}")

    yld_reg["Year"] = pd.to_numeric(yld_reg["Year"], errors="coerce")
    for col in TARGETS:
        yld_reg[col] = to_float(yld_reg[col])

    yld_reg = yld_reg.dropna(subset=["Year"] + TARGETS).astype({"Year": "int64"})

    os.makedirs(common_datasets_dir, exist_ok=True)

    #WideTable
    dup = df_filtered.duplicated(subset=["Year", "Month"]).sum()
    wide_aggfunc = "mean" if dup > 0 else "first"

    wide = df_filtered.pivot_table(
        index="Year",
        columns="Month",
        values=feat_cols,
        aggfunc=wide_aggfunc
    )
    wide.columns = [f"{feat}_m{int(m)}" for feat, m in wide.columns]
    wide = wide.reset_index()

    WideTable = wide.merge(yld_reg, on="Year", how="inner").sort_values("Year")
    wide_path = os.path.join(common_datasets_dir, f"{region_key}_WideTable.csv")
    WideTable.to_csv(wide_path, index=False, sep=SEP, encoding="utf-8-sig")

    #3PhaseTable
    phase1 = agg_mean_sum(df_filtered, months=[4, 5], suffix="_4_5", precip_col=PRECIP_COL)
    phase2 = agg_mean_sum(df_filtered, months=[6, 7], suffix="_6_7", precip_col=PRECIP_COL)
    phase3 = agg_mean_sum(df_filtered, months=[8, 9], suffix="_8_9", precip_col=PRECIP_COL)

    X_3phase = phase1.merge(phase2, on="Year", how="outer").merge(phase3, on="Year", how="outer")
    ThreePhaseTable = X_3phase.merge(yld_reg, on="Year", how="inner").sort_values("Year")
    three_path = os.path.join(common_datasets_dir, f"{region_key}_3PhaseTable.csv")
    ThreePhaseTable.to_csv(three_path, index=False, sep=SEP, encoding="utf-8-sig")

    #2PhaseTable
    phaseA = agg_mean_sum(df_filtered, months=[4, 5, 6], suffix="_4_6", precip_col=PRECIP_COL)
    phaseB = agg_mean_sum(df_filtered, months=[7, 8, 9], suffix="_7_9", precip_col=PRECIP_COL)

    X_2phase = phaseA.merge(phaseB, on="Year", how="outer")
    TwoPhaseTable = X_2phase.merge(yld_reg, on="Year", how="inner").sort_values("Year")
    two_path = os.path.join(common_datasets_dir, f"{region_key}_2PhaseTable.csv")
    TwoPhaseTable.to_csv(two_path, index=False, sep=SEP, encoding="utf-8-sig")

    #YearTable
    YearAgg = agg_mean_sum(df_filtered, months=[4, 5, 6, 7, 8, 9], suffix="_year", precip_col=PRECIP_COL)
    YearTable = YearAgg.merge(yld_reg, on="Year", how="inner").sort_values("Year")
    year_path = os.path.join(common_datasets_dir, f"{region_key}_YearTable.csv")
    YearTable.to_csv(year_path, index=False, sep=SEP, encoding="utf-8-sig")

    return {
        "YearTable": year_path,
        "WideTable": wide_path,
        "2PhaseTable": two_path,
        "3PhaseTable": three_path
    }


def run_dataset(file_path: str, tag: str, region_key: str, out_dir: str, 
                feature_log: list, cv_mode: str, min_train: int, n_splits: int, test_size: int):
    
    df = pd.read_csv(file_path, sep=SEP)
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

    for t in TARGETS:
        df[t] = to_float(df[t])

    df = df.dropna(subset=["Year"] + TARGETS).sort_values("Year").reset_index(drop=True)
    df["Year"] = df["Year"].astype(int)

    feature_cols = [c for c in df.columns if c not in ["Year"] + TARGETS]
    for c in feature_cols:
        df[c] = to_float(df[c])

    splits = get_time_splits(len(df), cv_mode, min_train, n_splits, test_size)
    if len(splits) == 0:
        print(f"[WARN] {tag}: not enough data for CV. len={len(df)}; cv_mode={cv_mode}")
        return pd.DataFrame(columns=["cv_mode", "dataset", "target", "model", "R2", "RMSE", "MAE"])

    df_num = df[feature_cols + TARGETS].copy()
    for target in TARGETS:
        corr_all = df_num[feature_cols + [target]].corr(method="spearman")
        rho = corr_all.loc[feature_cols, target].dropna()

        heat_path = os.path.join(out_dir, f"{safe_filename(tag)}_{safe_filename(target)}_heatmap_all.png")
        save_heatmap(corr_all, f"{tag}: Spearman heatmap (features + {target})", heat_path)

        bar_path = os.path.join(out_dir, f"{safe_filename(tag)}_{safe_filename(target)}_bar_absrho_top25.png")
        save_bar_absrho(rho, f"{tag}: Top-|ρ| features vs {target}", bar_path, topn=25)

        prefix = os.path.join(out_dir, f"{safe_filename(tag)}_{safe_filename(target)}")
        save_scatter_top(df_num, rho, target, prefix, topk=6)

    oof_preds = []
    feature_counter = {t: Counter() for t in TARGETS}

    for target in TARGETS:
        for fold, (tr, te) in enumerate(splits):
            train, test = df.iloc[tr], df.iloc[te]

            X_tr, X_te = train[feature_cols], test[feature_cols]
            y_tr, y_te = train[target], test[target]

            selected, _rho = spearman_select(X_tr, y_tr)
            feature_counter[target].update(selected)

            feature_log.append({
                "region": region_key,
                "dataset": tag,
                "target": target,
                "cv_mode": cv_mode,
                "fold": int(fold),
                "test_year": int(test["Year"].iloc[0]),
                "n_features": int(len(selected)),
                "features": ";".join(selected)
            })

            X_tr_sel = X_tr[selected]
            X_te_sel = X_te[selected]

            models = build_models(selected)

            # baselines
            y_mean = y_tr.mean()
            y_lag = y_tr.iloc[-1]

            for name, model in models.items():
                model.fit(X_tr_sel, y_tr)
                y_pred = model.predict(X_te_sel)
                for yr, yt, yp in zip(test["Year"], y_te, y_pred):
                    oof_preds.append({
                        "dataset": tag, "target": target, "fold": fold, "year": int(yr),
                        "model": name, "y_true": float(yt), "y_pred": float(yp)
                    })

            for yr, yt in zip(test["Year"], y_te):
                oof_preds.append({
                    "dataset": tag, "target": target, "fold": fold, "year": int(yr),
                    "model": "baseline_mean", "y_true": float(yt), "y_pred": float(y_mean)
                })
                oof_preds.append({
                    "dataset": tag, "target": target, "fold": fold, "year": int(yr),
                    "model": "baseline_lag1", "y_true": float(yt), "y_pred": float(y_lag)
                })

    oof_df = pd.DataFrame(oof_preds)

    metrics = []
    for (target, model), g in oof_df.groupby(["target", "model"]):
        metrics.append({
            "cv_mode": cv_mode,
            "dataset": tag,
            "target": target,
            "model": model,
            "R2": r2_score(g["y_true"], g["y_pred"]),
            "RMSE": rmse(g["y_true"], g["y_pred"]),
            "MAE": mean_absolute_error(g["y_true"], g["y_pred"]),
        })

    res_df = pd.DataFrame(metrics).sort_values(["target", "RMSE"])

    res_df.to_csv(os.path.join(out_dir, f"{safe_filename(tag)}_metrics.csv"),
                  sep=SEP, index=False, encoding="utf-8-sig")
    oof_df.to_csv(os.path.join(out_dir, f"{safe_filename(tag)}_oof_predictions.csv"),
                  sep=SEP, index=False, encoding="utf-8-sig")

    for t in TARGETS:
        stab = pd.DataFrame(feature_counter[t].most_common(), columns=["feature", "count"])
        stab.to_csv(os.path.join(out_dir, f"{safe_filename(tag)}_feature_stability_{safe_filename(t)}.csv"),
                    sep=SEP, index=False, encoding="utf-8-sig")

    return res_df


def run_region(region_key: str, workdir: str, common_datasets_dir: str, out_base: str, feature_log: list, 
               cv_mode: str, min_train: int, n_splits: int, test_size: int) -> pd.DataFrame:
    
    print(f"\n{'='*50}")
    print(f"REGION: {region_key} | CV_MODE: {cv_mode}")
    print(f"{'='*50}")

    paths = build_tables_for_region(region_key, workdir, common_datasets_dir)

    out_dir = os.path.join(out_base, f"model_results_{safe_filename(region_key)}")
    os.makedirs(out_dir, exist_ok=True)

    all_results = []
    for ds_name, fp in paths.items():
        tag = f"{region_key}_{ds_name}"
        all_results.append(run_dataset(fp, tag, region_key, out_dir, feature_log,
                                       cv_mode, min_train, n_splits, test_size))

    summary = pd.concat(all_results, axis=0).reset_index(drop=True)
    summary.to_csv(os.path.join(out_dir, f"{safe_filename(region_key)}_SUMMARY.csv"),
                   sep=SEP, index=False, encoding="utf-8-sig")
    return summary


def plot_multicollinearity_heatmaps(regions: list, common_datasets_dir: str, common_plots_dir: str):
    
    os.makedirs(common_plots_dir, exist_ok=True)
    region_corr = {}
    
    for region_key in regions:
        year_path = os.path.join(common_datasets_dir, f"{region_key}_YearTable.csv")
        if not os.path.exists(year_path):
            print(f"[WARN] Пропущен {region_key}: нет {year_path}")
            continue

        region_en = region_to_en(region_key)
        df = pd.read_csv(year_path, sep=SEP)
        
        drop_cols = ["Year"] + TARGETS
        feat_cols = [c for c in df.columns if c not in drop_cols]
        X = df[feat_cols].copy()
        
        for c in X.columns:
            X[c] = to_float(X[c])
        
        X = X.dropna(axis=1, how="all")
        
        if X.shape[1] < 2:
            print(f"[WARN] {region_en}: not enough features ({X.shape[1]})")
            continue

        C = X.corr(method="spearman")
        
        A = C.abs().copy()
        np.fill_diagonal(A.values, np.nan)
        score = A.mean(axis=1).sort_values(ascending=False)
        top_feats = score.head(TOPN_FEATURES).index.tolist()
        
        C_top = C.loc[top_feats, top_feats]
        region_corr[region_en] = C_top

        fig = plt.figure(figsize=(8, 7))
        ax = plt.gca()
        im = ax.imshow(C_top.values, aspect="auto", vmin=-1, vmax=1, cmap='RdBu_r')
        ax.set_title(f"Spearman feature correlation — {region_en}")
        ax.set_xticks(range(len(top_feats)))
        ax.set_yticks(range(len(top_feats)))
        ax.set_xticklabels(top_feats, rotation=90, fontsize=7)
        ax.set_yticklabels(top_feats, fontsize=7)
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label("Spearman ρ", rotation=90)
        plt.tight_layout()
        
        out_png = os.path.join(common_plots_dir, f"MULTICOL_SPEARMAN_YearTable_{safe_filename(region_en)}_top{TOPN_FEATURES}.png")
        plt.savefig(out_png, dpi=FIG_DPI)
        plt.close()

    preferred_order = ["Arshaly", "Ereimentau", "Zhaksy", "Korgalzhyn"]
    ordered = [r for r in preferred_order if r in region_corr] + [r for r in region_corr if r not in preferred_order]

    if len(ordered) > 0:
        n = len(ordered)
        nrows, ncols = (2, 2) if n == 4 else (int(np.ceil(n/2)), 2)
        figsize = (14, 10) if n == 4 else (14, 4 + 4*nrows)

        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize)
        axes = np.array(axes).reshape(-1)

        last_im = None
        for i, region_en in enumerate(ordered):
            ax = axes[i]
            C_top = region_corr[region_en]
            last_im = ax.imshow(C_top.values, aspect="auto", vmin=-1, vmax=1, cmap='RdBu_r')
            ax.set_title(region_en)
            ax.set_xticks(range(C_top.shape[1]))
            ax.set_yticks(range(C_top.shape[0]))
            ax.set_xticklabels(C_top.columns, rotation=90, fontsize=7)
            ax.set_yticklabels(C_top.index, fontsize=7)

        for j in range(i+1, len(axes)):
            axes[j].axis("off")

        cbar = fig.colorbar(last_im, ax=axes.tolist(), fraction=0.02, pad=0.02)
        cbar.set_label("Spearman ρ", rotation=90)

        fig.suptitle(f"Feature-feature correlation (YearTable): top {TOPN_FEATURES} features", fontsize=14)
        plt.tight_layout(rect=[0, 0, 1, 0.95])

        out_png_grid = os.path.join(common_plots_dir, f"MULTICOL_SPEARMAN_YearTable_ALLREGIONS_2x2_top{TOPN_FEATURES}.png")
        plt.savefig(out_png_grid, dpi=FIG_DPI)
        plt.close()

        print(f"\n[OK] Multicollinearity graphs are saved in {common_plots_dir}")


def plot_multicollinearity_stats(regions: list, common_datasets_dir: str, common_plots_dir: str):
    
    os.makedirs(common_plots_dir, exist_ok=True)
    thresholds = [0.7, 0.8, 0.9]
    stats = []
    abs_corr_vals = {}

    for region_key in regions:
        year_path = os.path.join(common_datasets_dir, f"{region_key}_YearTable.csv")
        if not os.path.exists(year_path):
            continue

        region_en = region_to_en(region_key)
        df = pd.read_csv(year_path, sep=SEP)
        
        drop_cols = ["Year"] + TARGETS
        feat_cols = [c for c in df.columns if c not in drop_cols]
        X = df[feat_cols].copy()
        
        for c in X.columns:
            X[c] = to_float(X[c])
        
        X = X.dropna(axis=1, how="all")
        
        if X.shape[1] < 2:
            continue

        C = X.corr(method="spearman")
        A = C.abs().values
        iu = np.triu_indices_from(A, k=1)
        vals = A[iu]
        vals = vals[~np.isnan(vals)]
        
        abs_corr_vals[region_en] = vals

        row = {
            "region": region_en,
            "n_features": int(X.shape[1]),
            "n_pairs": int(len(vals))
        }
        for t in thresholds:
            row[f"share_|rho|>={t}"] = float(np.mean(vals >= t)) if len(vals) else np.nan
        row["mean_|rho|"] = float(np.mean(vals)) if len(vals) else np.nan
        row["median_|rho|"] = float(np.median(vals)) if len(vals) else np.nan
        stats.append(row)

    stats_df = pd.DataFrame(stats).sort_values("region")
    stats_df.to_csv(os.path.join(common_plots_dir, "MULTICOL_SPEARMAN_YearTable_pairwise_stats.csv"),
                    sep=SEP, index=False, encoding="utf-8-sig")

    preferred_order = ["Arshaly", "Ereimentau", "Zhaksy", "Korgalzhyn"]
    ordered = [r for r in preferred_order if r in abs_corr_vals] + [r for r in abs_corr_vals if r not in preferred_order]

    if len(ordered) > 0:
        n = len(ordered)
        nrows, ncols = (2, 2) if n == 4 else (int(np.ceil(n/2)), 2)
        figsize = (14, 10) if n == 4 else (14, 4 + 4*nrows)

        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize)
        axes = np.array(axes).reshape(-1)

        for i, r in enumerate(ordered):
            ax = axes[i]
            vals = abs_corr_vals[r]
            ax.hist(vals, bins=30, edgecolor='black')
            ax.set_title(r)
            ax.set_xlabel("|Spearman ρ|")
            ax.set_ylabel("Count")
            
            for t in thresholds:
                ax.axvline(t, linestyle="--", linewidth=1, alpha=0.7)

        for j in range(i+1, len(axes)):
            axes[j].axis("off")

        fig.suptitle("Distribution of |Spearman ρ| across ALL feature pairs", fontsize=14)
        plt.tight_layout(rect=[0, 0, 1, 0.95])

        out_hist = os.path.join(common_plots_dir, "MULTICOL_SPEARMAN_YearTable_ALLREGIONS_hist_pairs.png")
        plt.savefig(out_hist, dpi=FIG_DPI)
        plt.close()

    if not stats_df.empty:
        x = np.arange(len(stats_df))
        width = 0.22

        plt.figure(figsize=(12, 6))
        for k, t in enumerate(thresholds):
            plt.bar(x + (k - 1) * width, stats_df[f"share_|rho|>={t}"].values, 
                   width, label=f"|ρ| ≥ {t}")

        plt.xticks(x, stats_df["region"].values, rotation=0)
        plt.ylim(0, 1)
        plt.ylabel("Share of feature pairs")
        plt.title("Share of strongly correlated climate feature pairs (YearTable)")
        plt.legend()
        plt.tight_layout()

        out_share = os.path.join(common_plots_dir, "MULTICOL_SPEARMAN_YearTable_ALLREGIONS_share_thresholds.png")
        plt.savefig(out_share, dpi=FIG_DPI)
        plt.close()

    print(f"[OK] multicollinearity statistics saved in {common_plots_dir}")


def run_analysis_for_mode(cv_mode: str, output_folder: str, common_datasets_dir: str):
    
    
    print(f"\nANALYSIS IN MODE: {cv_mode}")
    print(f"OUTPUT: {output_folder}")
    

    os.makedirs(output_folder, exist_ok=True)

    os.chdir(WORKDIR)
    final_files = sorted(glob.glob("*_Final.csv"))
    if not final_files:
        raise FileNotFoundError("Files not found - *_Final.csv в WORKDIR")

    regions = [re.sub(r"_Final\.csv$", "", f) for f in final_files]
    print(f"districts found: {regions}\n")

    if cv_mode == "rolling":
        min_train_val = MIN_TRAIN
        n_splits_val = None
    else:  # tscv
        min_train_val = None
        n_splits_val = N_SPLITS

    FEATURE_LOG = []
    ALL = []

    for region_key in regions:
        try:
            summ = run_region(
                region_key=region_key,
                workdir=WORKDIR,
                common_datasets_dir=common_datasets_dir,
                out_base=output_folder,
                feature_log=FEATURE_LOG,
                cv_mode=cv_mode,
                min_train=min_train_val if cv_mode == "rolling" else 0,
                n_splits=n_splits_val if cv_mode == "tscv" else 0,
                test_size=TEST_SIZE
            )
            summ.insert(0, "region", region_key)
            ALL.append(summ)
        except Exception as e:
            print(f"[ERROR] {region_key}: {e}")

    ALL_SUMMARY = pd.concat(ALL, axis=0).reset_index(drop=True) if ALL else pd.DataFrame()

    out_summary = os.path.join(output_folder, f"ALL_REGIONS_SUMMARY_{cv_mode}.csv")
    out_log = os.path.join(output_folder, f"ALL_REGIONS_SELECTED_FEATURES_LOG_{cv_mode}.csv")

    ALL_SUMMARY.to_csv(out_summary, sep=SEP, index=False, encoding="utf-8-sig")
    pd.DataFrame(FEATURE_LOG).to_csv(out_log, sep=SEP, index=False, encoding="utf-8-sig")

    print(f"\nDone for {cv_mode}:")
    print(f"  Total for all districts: {out_summary}")
    print(f"  features logs: {out_log}")
    
    return regions



if __name__ == "__main__":
    
    os.makedirs(COMMON_PLOTS_DIR, exist_ok=True)
    os.makedirs(COMMON_DATASETS_DIR, exist_ok=True)
    os.makedirs(COMMON_RESULTS_DIR, exist_ok=True)
    
    print("\nStructure:")
    print(f"  Common datasets: {COMMON_DATASETS_DIR}")
    print(f"  Common diagram:  {COMMON_PLOTS_DIR}")
    print(f"  Common results: {COMMON_RESULTS_DIR}")
        
    regions_tscv = run_analysis_for_mode(
        cv_mode="tscv",
        output_folder=os.path.join(WORKDIR, "regression_tscv"),
        common_datasets_dir=COMMON_DATASETS_DIR
    )
    
    regions_rolling = run_analysis_for_mode(
        cv_mode="rolling",
        output_folder=os.path.join(WORKDIR, "regression_rolling"),
        common_datasets_dir=COMMON_DATASETS_DIR
    )
    
    regions = regions_tscv if regions_tscv else regions_rolling
    if regions:
        plot_multicollinearity_heatmaps(regions, COMMON_DATASETS_DIR, COMMON_PLOTS_DIR)
        plot_multicollinearity_stats(regions, COMMON_DATASETS_DIR, COMMON_PLOTS_DIR)
    
    f_tscv = os.path.join(WORKDIR, "regression_tscv", "ALL_REGIONS_SUMMARY_tscv.csv")
    f_rolling = os.path.join(WORKDIR, "regression_rolling", "ALL_REGIONS_SUMMARY_rolling.csv")
    f_combined = os.path.join(COMMON_RESULTS_DIR, "ALL_REGIONS_SUMMARY_COMBINED.csv")
    
    if os.path.exists(f_tscv) and os.path.exists(f_rolling):
        tscv_df = pd.read_csv(f_tscv, sep=SEP)
        rolling_df = pd.read_csv(f_rolling, sep=SEP)
        
        if "cv_mode" not in tscv_df.columns:
            tscv_df.insert(0, "cv_mode", "tscv")
        if "cv_mode" not in rolling_df.columns:
            rolling_df.insert(0, "cv_mode", "rolling")
        
        combined = pd.concat([tscv_df, rolling_df], ignore_index=True)
        combined.to_csv(f_combined, sep=SEP, index=False, encoding="utf-8-sig")
        
        print(f"[OK] common table saved: {f_combined}")
    else:
        print("[WARN] Both files could not be combined")
    
    print("\nDone:")
    print(f"Multicollinearity graphs: {COMMON_PLOTS_DIR}")
    print(f"Datasets by district: {COMMON_DATASETS_DIR}")
    print(f"Common results: {COMMON_RESULTS_DIR}")
    print(f"TSCV result: {os.path.join(WORKDIR, 'regression_tscv')}")
    print(f"Rolling result: {os.path.join(WORKDIR, 'regression_rolling')}")
    


Structure:
  Common datasets: C:\kazhydromet\regression_common_datasets
  Common diagram:  C:\kazhydromet\regression_common_plots
  Common results: C:\kazhydromet\regression_common_results

ANALYSIS IN MODE: tscv
OUTPUT: C:\kazhydromet\regression_tscv
districts found: ['Аршалы', 'Ерейментау', 'Жаксы', 'Коргалжын']


REGION: Аршалы | CV_MODE: tscv
[Аршалы]  7 multicollinear features deleted 

REGION: Ерейментау | CV_MODE: tscv
[Ерейментау]  7 multicollinear features deleted 

REGION: Жаксы | CV_MODE: tscv
[Жаксы]  6 multicollinear features deleted 

REGION: Коргалжын | CV_MODE: tscv
[Коргалжын]  7 multicollinear features deleted 

Done for tscv:
  Total for all districts: C:\kazhydromet\regression_tscv\ALL_REGIONS_SUMMARY_tscv.csv
  features logs: C:\kazhydromet\regression_tscv\ALL_REGIONS_SELECTED_FEATURES_LOG_tscv.csv

ANALYSIS IN MODE: rolling
OUTPUT: C:\kazhydromet\regression_rolling
districts found: ['Аршалы', 'Ерейментау', 'Жаксы', 'Коргалжын']


REGION: Аршалы | CV_MODE: rolling

C:\Users\TIMING\AppData\Local\Temp\ipykernel_4532\2096117403.py:583: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])



[OK] Multicollinearity graphs are saved in C:\kazhydromet\regression_common_plots
[OK] multicollinearity statistics saved in C:\kazhydromet\regression_common_plots
[OK] common table saved: C:\kazhydromet\regression_common_results\ALL_REGIONS_SUMMARY_COMBINED.csv

Done:
Multicollinearity graphs: C:\kazhydromet\regression_common_plots
Datasets by district: C:\kazhydromet\regression_common_datasets
Common results: C:\kazhydromet\regression_common_results
TSCV result: C:\kazhydromet\regression_tscv
Rolling result: C:\kazhydromet\regression_rolling
